In [100]:
import pandas as pd

print("Start cleaning batch")

CSV_PATH = "data1.csv"
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=["", "NA", "NaN"])

print(f"Loaded rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print(df.head())

Start cleaning batch
Loaded rows: 346,018
Columns (31): ['Creation Date', 'Purchase Date', 'Fiscal Year', 'LPA Number', 'Purchase Order Number', 'Requisition Number', 'Acquisition Type', 'Sub-Acquisition Type', 'Acquisition Method', 'Sub-Acquisition Method', 'Department Name', 'Supplier Code', 'Supplier Name', 'Supplier Qualifications', 'Supplier Zip Code', 'CalCard', 'Item Name', 'Item Description', 'Quantity', 'Unit Price', 'Total Price', 'Classification Codes', 'Normalized UNSPSC', 'Commodity Title', 'Class', 'Class Title', 'Family', 'Family Title', 'Segment', 'Segment Title', 'Location']
  Creation Date Purchase Date Fiscal Year   LPA Number Purchase Order Number  \
0    08/27/2013           NaN   2013-2014   7-12-70-26            REQ0011118   
1    01/29/2014           NaN   2013-2014          NaN            REQ0011932   
2    11/01/2013           NaN   2013-2014          NaN            REQ0011476   
3    06/13/2014    06/05/2014   2013-2014          NaN            4500236642   
4

In [101]:
print("\n=== coulms ==")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")


=== coulms ==
1. Creation Date
2. Purchase Date
3. Fiscal Year
4. LPA Number
5. Purchase Order Number
6. Requisition Number
7. Acquisition Type
8. Sub-Acquisition Type
9. Acquisition Method
10. Sub-Acquisition Method
11. Department Name
12. Supplier Code
13. Supplier Name
14. Supplier Qualifications
15. Supplier Zip Code
16. CalCard
17. Item Name
18. Item Description
19. Quantity
20. Unit Price
21. Total Price
22. Classification Codes
23. Normalized UNSPSC
24. Commodity Title
25. Class
26. Class Title
27. Family
28. Family Title
29. Segment
30. Segment Title
31. Location


In [102]:

print("\n=== summery ===")
summary = []
for col in df.columns:
    non_null = df[col].notna().sum()
    nulls = df[col].isna().sum()
    unique_vals = df[col].nunique(dropna=True)
    sample_values = df[col].dropna().unique()[:3]
    summary.append({
        "coulm": col,
        "non_null": non_null,
        "null": nulls,
        "unique_vals": unique_vals,
    })

summary_df = pd.DataFrame(summary)
print(summary_df)


=== summery ===
                      coulm  non_null    null  unique_vals
0             Creation Date    346018       0         1015
1             Purchase Date    328582   17436         2268
2               Fiscal Year    346018       0            3
3                LPA Number     92345  253673         1420
4     Purchase Order Number    346018       0       200533
5        Requisition Number     14369  331649         5997
6          Acquisition Type    346018       0            5
7      Sub-Acquisition Type     68337  277681           25
8        Acquisition Method    346018       0           20
9    Sub-Acquisition Method     30896  315122           16
10          Department Name    346018       0          111
11            Supplier Code    345982      36        25239
12            Supplier Name    345982      36        24732
13  Supplier Qualifications    141745  204273          278
14        Supplier Zip Code    275908   70110         3993
15                  CalCard    346018  

In [103]:
# Info about nulls and dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346018 entries, 0 to 346017
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Creation Date            346018 non-null  object
 1   Purchase Date            328582 non-null  object
 2   Fiscal Year              346018 non-null  object
 3   LPA Number               92345 non-null   object
 4   Purchase Order Number    346018 non-null  object
 5   Requisition Number       14369 non-null   object
 6   Acquisition Type         346018 non-null  object
 7   Sub-Acquisition Type     68337 non-null   object
 8   Acquisition Method       346018 non-null  object
 9   Sub-Acquisition Method   30896 non-null   object
 10  Department Name          346018 non-null  object
 11  Supplier Code            345982 non-null  object
 12  Supplier Name            345982 non-null  object
 13  Supplier Qualifications  141745 non-null  object
 14  Supplier Zip Code   

In [104]:
import pandas as pd

def to_dt_safe(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT
    s = str(x).strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(s, format=fmt, errors="raise")
        except Exception:
            pass
    return pd.to_datetime(s, errors="coerce", dayfirst=False)

date_cols = ["Creation Date", "Purchase Date"]

for c in date_cols:
    df[c] = df[c].apply(to_dt_safe)

print(df[date_cols].dtypes)


Creation Date    datetime64[ns]
Purchase Date    datetime64[ns]
dtype: object


In [105]:
import pandas as pd
import numpy as np


df["CreationDate_dt"] = pd.to_datetime(df["Creation Date"], errors="coerce")


def fiscal_year_from_date(dt):
    if pd.isna(dt):
        return None
    year = dt.year

    if dt.month >= 7:
        return f"{year}-{year+1}"
    else:
        return f"{year-1}-{year}"

df["FiscalYear_calc"] = df["CreationDate_dt"].apply(fiscal_year_from_date)


df["FiscalYear_norm"] = df["Fiscal Year"].astype(str).str.replace("–", "-").str.replace("—", "-").str.strip()


df["FY_match"] = df["FiscalYear_calc"] == df["FiscalYear_norm"]

total = len(df)
matches = df["FY_match"].sum()
mismatch = total - matches


print(f"total row {total:,}")
print(f"identical: {matches:,} ({matches/total:.2%})")
print(f"different or missing: {mismatch:,} ({mismatch/total:.2%})")


print("\n===  sample ===")
print(df.loc[~df["FY_match"], ["Creation Date", "Fiscal Year", "FiscalYear_calc"]].head(10))


total row 346,018
identical: 346,018 (100.00%)
different or missing: 0 (0.00%)

===  sample ===
Empty DataFrame
Columns: [Creation Date, Fiscal Year, FiscalYear_calc]
Index: []


In [106]:

df["Purchase Date"] = df["Purchase Date"].replace(r'^\s*$', pd.NA, regex=True)


df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")


df["Purchase Date"].isna().sum(), df["Purchase Date"].shape


(np.int64(17462), (346018,))

In [107]:

print("\n=== ===")
summary = []
for col in df.columns:
    non_null = df[col].notna().sum()
    nulls = df[col].isna().sum()
    unique_vals = df[col].nunique(dropna=True)
    sample_values = df[col].dropna().unique()[:3]  # أول 3 قيم مميزة كمثال
    summary.append({
        "coulm": col,
        "non_null": non_null,
        "null": nulls,
        "unique_vals": unique_vals,
    })

summary_df = pd.DataFrame(summary)
print(summary_df)


=== ===
                      coulm  non_null    null  unique_vals
0             Creation Date    346018       0         1015
1             Purchase Date    328556   17462         2253
2               Fiscal Year    346018       0            3
3                LPA Number     92345  253673         1420
4     Purchase Order Number    346018       0       200533
5        Requisition Number     14369  331649         5997
6          Acquisition Type    346018       0            5
7      Sub-Acquisition Type     68337  277681           25
8        Acquisition Method    346018       0           20
9    Sub-Acquisition Method     30896  315122           16
10          Department Name    346018       0          111
11            Supplier Code    345982      36        25239
12            Supplier Name    345982      36        24732
13  Supplier Qualifications    141745  204273          278
14        Supplier Zip Code    275908   70110         3993
15                  CalCard    346018       0  

In [108]:
df["LPA Number"] = (
    df["LPA Number"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None, "Not Applicable": None})
)


In [109]:
df["Requisition Number"] = (
    df["Requisition Number"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [110]:
import re
def clean_value(x):
    if pd.isna(x):
        return ""
    x = re.sub(r'[\u00A0\u200B\u200C\u200D\uFEFF]', '', str(x))
    return x.strip().lower()

df["Purchase Order Number"] = df["Purchase Order Number"].apply(clean_value)
df["Requisition Number"] = df["Requisition Number"].apply(clean_value)



In [111]:
df["Sub-Acquisition Type"] = (
    df["Sub-Acquisition Type"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [112]:
df["Sub-Acquisition Method"] = (
    df["Sub-Acquisition Method"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [113]:
import re

# Count nulls before cleaning
before_code_nulls = df["Supplier Code"].isna().sum()
before_name_nulls = df["Supplier Name"].isna().sum()

# Clean Supplier Code → Only keep real codes
def clean_supplier_code(x):
    x = str(x).strip()
    if x.lower() in ["", "nan", "none", "null", "-", "0", "000", "0000", "00000"]:
        return None
    return x  # Keep raw format

# Clean Supplier Name → Only remove placeholders, keep original casing
def clean_supplier_name(x):
    x = str(x).strip()
    if x.lower() in ["", "nan", "none", "null", "unknown", "-"]:
        return None
    return x  # No title formatting applied

# Apply cleaning
df["Supplier Code"] = df["Supplier Code"].apply(clean_supplier_code)
df["Supplier Name"] = df["Supplier Name"].apply(clean_supplier_name)

# Count nulls after cleaning
after_code_nulls = df["Supplier Code"].isna().sum()
after_name_nulls = df["Supplier Name"].isna().sum()

# Print conversion stats
print(f"✅ Supplier Code → Converted to None: {after_code_nulls - before_code_nulls}")
print(f"✅ Supplier Name → Converted to None: {after_name_nulls - before_name_nulls}")


✅ Supplier Code → Converted to None: 4473
✅ Supplier Name → Converted to None: 4473


In [114]:
import pandas as pd


code_to_name_check = (
    df.groupby("Supplier Code")["Supplier Name"]
      .nunique()
      .reset_index(name="unique_names_per_code")
)


name_to_code_check = (
    df.groupby("Supplier Name")["Supplier Code"]
      .nunique()
      .reset_index(name="unique_codes_per_name")
)


codes_with_multiple_names = code_to_name_check[code_to_name_check["unique_names_per_code"] > 1]
names_with_multiple_codes = name_to_code_check[name_to_code_check["unique_codes_per_name"] > 1]

print("🔍 Codes linked to multiple different supplier names:")
print(codes_with_multiple_names.head(20))

print("\n🔍 Supplier names linked to multiple different codes:")
print(names_with_multiple_codes.head(26))
print(len(names_with_multiple_codes))



🔍 Codes linked to multiple different supplier names:
Empty DataFrame
Columns: [Supplier Code, unique_names_per_code]
Index: []

🔍 Supplier names linked to multiple different codes:
                          Supplier Name  unique_codes_per_name
55                      3W Construction                      2
122      A Cut Above Tree Service, Inc.                      2
142            A TO Z BUILDING SERVICES                      2
161                      A&R Provisions                      2
189                 A-Z Bus Sales, Inc.                      2
239                      ABC Sanitation                      2
279             ACCO Engineered Systems                      2
289                        ACCUVANT INC                      2
305                   ACL Services Ltd.                      2
369                               AECOM                      2
371      AECOM Technical Services, Inc.                      2
546   AMEC Environment & Infrastructure                      3


In [115]:
import pandas as pd


df["Supplier Qualifications"] = (
    df["Supplier Qualifications"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
    .apply(lambda x: [val.strip() for val in str(x).split() if val.strip()] if pd.notna(x) else None)
)


In [116]:
print(df["Supplier Qualifications"])

0                   None
1                   None
2                   None
3         [CA-MB, CA-SB]
4                   None
               ...      
346013           [CA-SB]
346014              None
346015    [CA-MB, CA-SB]
346016    [CA-MB, CA-SB]
346017              None
Name: Supplier Qualifications, Length: 346018, dtype: object


In [117]:
df["Supplier Zip Code"] = (
    df["Supplier Zip Code"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [118]:
print(df["Supplier Zip Code"])

0          None
1          None
2         95841
3         91436
4         95814
          ...  
346013    95811
346014    95816
346015    95829
346016    95630
346017    96007
Name: Supplier Zip Code, Length: 346018, dtype: object


In [119]:
print(df["CalCard"])

0         NO
1         NO
2         NO
3         NO
4         NO
          ..
346013    NO
346014    NO
346015    NO
346016    NO
346017    NO
Name: CalCard, Length: 346018, dtype: object


In [120]:
# إذا كانت القيم بالفعل YES/NO بلا مسافات ولا اختلاف حالة:
df["CalCard"] = df["CalCard"].map({"YES": True, "NO": False})


In [121]:
print(df["CalCard"])

0         False
1         False
2         False
3         False
4         False
          ...  
346013    False
346014    False
346015    False
346016    False
346017    False
Name: CalCard, Length: 346018, dtype: bool


In [122]:
cols_to_check = [
    "Item Name", "Item Description", "Quantity", "Unit Price", "Total Price",
    "Classification Codes", "Normalized UNSPSC", "Commodity Title", "Class",
    "Class Title", "Family", "Family Title", "Segment", "Segment Title"
]

mask_empty = df[cols_to_check].isna().all(axis=1)
rows_to_drop = mask_empty.sum()

print(f"empty totally {rows_to_drop}")


df = df[~mask_empty].reset_index(drop=True)


empty totally 30


In [123]:
print(df.shape)


(345988, 35)


In [124]:
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

Columns (35): ['Creation Date', 'Purchase Date', 'Fiscal Year', 'LPA Number', 'Purchase Order Number', 'Requisition Number', 'Acquisition Type', 'Sub-Acquisition Type', 'Acquisition Method', 'Sub-Acquisition Method', 'Department Name', 'Supplier Code', 'Supplier Name', 'Supplier Qualifications', 'Supplier Zip Code', 'CalCard', 'Item Name', 'Item Description', 'Quantity', 'Unit Price', 'Total Price', 'Classification Codes', 'Normalized UNSPSC', 'Commodity Title', 'Class', 'Class Title', 'Family', 'Family Title', 'Segment', 'Segment Title', 'Location', 'CreationDate_dt', 'FiscalYear_calc', 'FiscalYear_norm', 'FY_match']


In [125]:
df.drop(columns=["CreationDate_dt", "FiscalYear_calc", "FiscalYear_norm", "FY_match"], inplace=True, errors="ignore")


In [126]:
print(df.shape)

(345988, 31)


In [127]:
import pandas as pd

print("\n=== ملخص الأعمدة ===")

def make_hashable(x):
    if isinstance(x, list):
        return tuple(x)
    if isinstance(x, dict):
        return tuple(sorted(x.items()))
    return x

summary = []
for col in df.columns:
    if col == "Supplier Qualifications":
        continue

    s = df[col]
    non_null = s.notna().sum()
    nulls = s.isna().sum()
    unique_vals = pd.Series(s.dropna().map(make_hashable)).nunique(dropna=True)
    summary.append({
        "coulm": col,
        "non_null": non_null,
        "null": nulls,
        "unique_vals": unique_vals,
    })

summary_df = pd.DataFrame(summary)
print(summary_df)



=== ملخص الأعمدة ===
                     coulm  non_null    null  unique_vals
0            Creation Date    345988       0         1015
1            Purchase Date    328526   17462         2253
2              Fiscal Year    345988       0            3
3               LPA Number     92336  253652         1420
4    Purchase Order Number    345988       0       200479
5       Requisition Number    345988       0         5995
6         Acquisition Type    345988       0            5
7     Sub-Acquisition Type     68337  277651           25
8       Acquisition Method    345988       0           20
9   Sub-Acquisition Method     30895  315093           16
10         Department Name    345988       0          111
11           Supplier Code    341479    4509        25238
12           Supplier Name    341479    4509        24731
13       Supplier Zip Code    275881   70107         3993
14                 CalCard    345988       0            2
15               Item Name    345988       0      

In [128]:
import pandas as pd

col = "Item Name"

print(f"\n=== تشخيص {col} ===")
true_null = df[col].isna().sum()
print("true empty:", true_null)

null_like_set = {
    "", "N/A", "NONE", "NULL", "UNKNOWN"
}

mask_null_like = df[col].astype(str).str.strip().str.upper().isin(null_like_set)
print("empty", int(mask_null_like.sum()))

print("\nmost repeated")
print(df.loc[mask_null_like, col].astype(str).str.strip().str.upper().value_counts().head(10))



=== تشخيص Item Name ===
true empty: 0
empty 3

most repeated
Item Name
NONE    2
N/A     1
Name: count, dtype: int64


In [129]:
df["Item Name"] = df["Item Name"].replace(
    {"NONE": None, "N/A": None, "": None}
)


In [130]:
print("Null count after cleaning:", df["Item Name"].isna().sum())



Null count after cleaning: 0


In [131]:
import pandas as pd
import numpy as np

COL = "Item Description"


NULL_LIKE = {
    "", "NONE", "N/A", "NA", "NULL", "UNKNOWN"
}

# ===== دوال مساعدة =====
def is_null_like(val: str) -> bool:
    v = val.strip().upper()
    return v in NULL_LIKE

def normalize_to_none(val):

    if pd.isna(val):
        return None

    if isinstance(val, str):
        v = val.strip()
        return None if is_null_like(v) else v
    return val

def print_diagnostics(df: pd.DataFrame, col: str):
    s = df[col]
    print(f"\n===  ===")

    true_null_mask = s.isna()
    print("real empty", int(true_null_mask.sum()))

    null_like_mask = s.apply(lambda x: isinstance(x, str) and is_null_like(x))
    print("empty:", int(null_like_mask.sum()))
    if null_like_mask.any():
        print("\nأكثر القيم الشبيهة بالفراغ تكرارًا:")
        print(
            s[null_like_mask]
            .astype(str).str.strip().str.upper()
            .value_counts()
            .head(20)
        )

print_diagnostics(df, COL)

_before = df[COL].copy()


df[COL] = df[COL].apply(normalize_to_none)


df[COL] = df[COL].where(df[COL].notna(), None)


changed_to_none_mask = _before.notna() & df[COL].isna()
num_changed = int(changed_to_none_mask.sum())

print(f"\n✅ none: {num_changed}")
if num_changed:
    print("\n before")
    print(
        _before[changed_to_none_mask]
        .astype(str).str.strip().str.upper()
        .value_counts()
        .head(20)
    )


print_diagnostics(df, COL)



===  ===
real empty 59
empty: 138

أكثر القيم الشبيهة بالفراغ تكرارًا:
Item Description
N/A     114
NONE     13
NA       11
Name: count, dtype: int64

✅ none: 138

 before
Item Description
N/A     114
NONE     13
NA       11
Name: count, dtype: int64

===  ===
real empty 197
empty: 0


In [132]:
import pandas as pd

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce").astype("float")


print(df["Quantity"].dtype)


float64


In [133]:
import pandas as pd
import re

def clean_price(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s == "":
        return None

    s = re.sub(r"[\$,]", "", s)

    s = s.replace("−", "-").replace("–", "-").replace("—", "-")
    try:
        return float(s)
    except ValueError:
        return None


for col in ["Unit Price", "Total Price"]:
    df[col] = df[col].apply(clean_price).astype("float")


print(df[["Unit Price", "Total Price"]].dtypes)



Unit Price     float64
Total Price    float64
dtype: object


In [134]:
import numpy as np


EPS = 0.01

# الأعمدة المختصرة
Q = df["Quantity"]
U = df["Unit Price"]
T = df["Total Price"]

# نحسب الفرق بين الناتج الفعلي والإجمالي المعلن
diff = (Q * U) - T

# تصنيف السطر بناءً على المنطق المالي
df["LineType"] = np.select(
    [
        (U < 0) & (T < 0),                                           # خصم أو تصحيح
        (U == 0) & (T > 0),                                          # رسوم أو ضريبة
        (U == 0) & (T == 0),                                         # تعديل عقد (بند بدون تكلفة)
        (np.round(diff, 2) != 0) & (abs(diff) <= EPS),               # فرق طفيف بعد التقريب ← تقريب
        (abs(diff) > EPS),                                           # فرق كبير ← غير متسق
    ],
    [
        "Discount",
        "FeeTax",
        "Amendment",
        "Rounding",
        "Inconsistent",
    ],
    default="Normal"
)


print("\n=== توزيع LineType ===")
print(df["LineType"].value_counts())




for cat in df["LineType"].unique():
    sample = df.loc[df["LineType"] == cat, ["Item Name", "Quantity", "Unit Price", "Total Price"]].head(3)
    print(f"\n--- أمثلة من {cat} ---")
    print(sample.to_string(index=False))



=== توزيع LineType ===
LineType
Normal          332614
Amendment         7518
Inconsistent      4220
Discount          1438
Rounding           164
FeeTax              34
Name: count, dtype: int64

--- أمثلة من Normal ---
    Item Name  Quantity  Unit Price  Total Price
          USB       1.0         1.0          1.0
Tire Disposal       2.0         2.0          4.0
        Labor       4.5       150.0        675.0

--- أمثلة من Inconsistent ---
         Item Name  Quantity  Unit Price  Total Price
         Fish Food   10000.0        0.48      4750.00
              Corn   10010.0        0.59      5855.85
EXT Trout Floating  120000.0        0.48     57000.00

--- أمثلة من Discount ---
     Item Name  Quantity  Unit Price  Total Price
      Discount       1.0       -7.00        -7.00
500GB 72000RPM       2.0      -76.44      -152.88
 Less Discount       1.0     -673.32      -673.32

--- أمثلة من Amendment ---
                Item Name  Quantity  Unit Price  Total Price
Stand, Read/Write, 

In [135]:
print(df.shape)

(345988, 32)


In [136]:
import pandas as pd

print("\n=== ملخص الأعمدة ===")

def make_hashable(x):
    if isinstance(x, list):
        return tuple(x)
    if isinstance(x, dict):
        return tuple(sorted(x.items()))
    return x

summary = []
for col in df.columns:
    if col == "Supplier Qualifications":
        continue

    s = df[col]
    non_null = s.notna().sum()
    nulls = s.isna().sum()
    unique_vals = pd.Series(s.dropna().map(make_hashable)).nunique(dropna=True)
    summary.append({
        "coulm": col,
        "non_null": non_null,
        "null": nulls,
        "unique_vals": unique_vals,
    })

summary_df = pd.DataFrame(summary)
print(summary_df)



=== ملخص الأعمدة ===
                     coulm  non_null    null  unique_vals
0            Creation Date    345988       0         1015
1            Purchase Date    328526   17462         2253
2              Fiscal Year    345988       0            3
3               LPA Number     92336  253652         1420
4    Purchase Order Number    345988       0       200479
5       Requisition Number    345988       0         5995
6         Acquisition Type    345988       0            5
7     Sub-Acquisition Type     68337  277651           25
8       Acquisition Method    345988       0           20
9   Sub-Acquisition Method     30895  315093           16
10         Department Name    345988       0          111
11           Supplier Code    341479    4509        25238
12           Supplier Name    341479    4509        24731
13       Supplier Zip Code    275881   70107         3993
14                 CalCard    345988       0            2
15               Item Name    345988       0      

In [137]:
import re
import pandas as pd

COL = "Classification Codes"


NULL_LIKE = {
    "", " ", "N/A", "NA", "NONE", "None", "NULL",
    "NOT APPLICABLE", "Not Applicable", "UNKNOWN"
}

def clean_classification_codes(cell):
    """تنظيف شامل لعمود Classification Codes"""

    if pd.isna(cell):
        return None

    s = str(cell).strip()
    if not s or s.upper() in NULL_LIKE:
        return None


    parts = re.split(r"[\s\n]+", s)
    codes = []

    for part in parts:

        digits = re.sub(r"\D", "", part)
        if not digits:
            continue


        if 1 <= len(digits) < 8:
            digits = digits.ljust(8, "0")


        elif len(digits) > 8:
            codes.extend(re.findall(r"\d{8}", digits))
            continue

        codes.append(digits)


    codes = list(dict.fromkeys(codes))
    return codes if codes else None



_before = df[COL].copy()
df[COL] = df[COL].apply(clean_classification_codes)


mask_none = df[COL].isna()
mask_list_multi = df[COL].apply(lambda v: isinstance(v, list) and len(v) > 1)
mask_list_single = df[COL].apply(lambda v: isinstance(v, list) and len(v) == 1)
mask_fixed_short = _before.astype(str).str.strip().str.match(r"^\d{1,7}$", na=False)



print(f"\n=== Cleanup Summary '{COL}' ===")
print(f"🔹 Number of final null values: {mask_none.sum():,}")
print(f"🔹 Number of codes that were missing and fixed: {mask_fixed_short.sum():,}")
print(f"🔹 Rows containing a single code: {mask_list_single.sum():,}")
print(f"🔹 Rows containing multiple codes: {mask_list_multi.sum():,}")


print("\n=== عينات بعد التنظيف ===")
sample = df.loc[~mask_none, [COL]].head(10)
for i, row in sample.iterrows():
    print(f"[{i}] → {row[COL]}")



=== Cleanup Summary 'Classification Codes' ===
🔹 Number of final null values: 987
🔹 Number of codes that were missing and fixed: 17
🔹 Rows containing a single code: 286,153
🔹 Rows containing multiple codes: 58,848

=== عينات بعد التنظيف ===
[1] → ['76121504']
[3] → ['44103127']
[4] → ['44103127']
[5] → ['85121615']
[6] → ['44103127']
[7] → ['40172800']
[10] → ['44103127']
[14] → ['50405625']
[15] → ['50301541']
[16] → ['55101506']


In [96]:
print(df.shape)

(345988, 32)


In [97]:
import pandas as pd


pairs = [
    ("Normalized UNSPSC", "Commodity Title"),
    ("Class", "Class Title"),
    ("Family", "Family Title"),
    ("Segment", "Segment Title"),
]

print("=== 🔍 (Code ↔ Title) ===")

for code_col, title_col in pairs:
    print(f"\n📘  {code_col} ↔ {title_col}")


    sub = df[[code_col, title_col]].dropna()


    mapping = sub.groupby(code_col)[title_col].nunique()

    total_codes = len(mapping)
    mismatched = (mapping > 1).sum()
    consistent = total_codes - mismatched

    print(f"🔹 uniqe: {total_codes:,}")
    print(f"✅ identical : {consistent:,}")
    print(f"⚠️ non identical: {mismatched:,}")


    if mismatched > 0:
        print("\n🧾 sample: ")
        bad_codes = mapping[mapping > 1].index[:10]
        for c in bad_codes:
            titles = sub.loc[sub[code_col] == c, title_col].dropna().unique().tolist()
            print(f"   {c}  →  {titles}")
    else:
        print("✅ ")

print("\n=== done")


=== 🔍 (Code ↔ Title) ===

📘  Normalized UNSPSC ↔ Commodity Title
🔹 uniqe: 13,296
✅ identical : 13,296
⚠️ non identical: 0
✅ 

📘  Class ↔ Class Title
🔹 uniqe: 2,363
✅ identical : 2,363
⚠️ non identical: 0
✅ 

📘  Family ↔ Family Title
🔹 uniqe: 409
✅ identical : 406
⚠️ non identical: 3

🧾 sample: 
   30100000  →  ['Structural components and basic shapes', 'Structural materials and basic shapes']
   31180000  →  ['Gaskets and seals', 'Packings glands boots and covers']
   72150000  →  ['Specialized trade construction and maintenance services', 'lized trade construction and maintenance services']

📘  Segment ↔ Segment Title
🔹 uniqe: 56
✅ identical : 56
⚠️ non identical: 0
✅ 

=== done


In [98]:
import pandas as pd


col_code = "Family"
col_title = "Family Title"


fix_map = {
    "30100000": "Structural materials and basic shapes",
    "31180000": "Gaskets and seals",
    "72150000": "Specialized trade construction and maintenance services",
}


print("=== before===")
mask_before = df[col_code].isin(fix_map.keys())
print(df.loc[mask_before, [col_code, col_title]].drop_duplicates().head(10))


df.loc[df[col_code].isin(fix_map.keys()), col_title] = (
    df.loc[df[col_code].isin(fix_map.keys()), col_code].map(fix_map)
)


print("\n=== after===")
print(df.loc[mask_before, [col_code, col_title]].drop_duplicates().head(10))


changed_count = df.loc[mask_before].shape[0]
print(f"\n✅ {changed_count:,} ")


=== before===
         Family                                       Family Title
60     31180000                                  Gaskets and seals
1132   30100000             Structural components and basic shapes
1479   72150000  Specialized trade construction and maintenance...
6147   30100000              Structural materials and basic shapes
6239   72150000  lized trade construction and maintenance services
32107  31180000                   Packings glands boots and covers

=== after===
        Family                                       Family Title
60    31180000                                  Gaskets and seals
1132  30100000              Structural materials and basic shapes
1479  72150000  Specialized trade construction and maintenance...

✅ 4,322 


In [138]:
import re
import pandas as pd

COL = "Normalized UNSPSC"

def clean_unspsc(cell):
    """تنظيف كود UNSPSC إلى الشكل القياسي 8 digits أو None"""
    if pd.isna(cell):
        return None

    s = str(cell).strip().upper()
    if not s or s in {"NONE", "N/A", "NULL", "NA", "NOT APPLICABLE", "UNKNOWN"}:
        return None

    digits = re.sub(r"\D", "", s)
    if not digits:
        return None


    if 1 <= len(digits) < 8:
        digits = digits.ljust(8, "0")


    elif len(digits) > 8:
        digits = digits[:8]

    return digits


_before = df[COL].copy()
df[COL] = df[COL].apply(clean_unspsc)


mask_none = df[COL].isna()
mask_fixed = _before.astype(str).str.strip().str.match(r"^\d{1,7}$", na=False)
mask_changed = (df[COL] != _before) & (~df[COL].isna())



print(f"\n=== Column Cleanup: {COL} ===")
print(f"🔹 Total Rows: {len(df):,}")
print(f"🔹 Number of Final Empty Values: {mask_none.sum():,}")
print(f"🔹 Number of Shortcodes Fixed (<8 digits): {mask_fixed.sum():,}")
print(f"🔹 Number of Values ​​Actually Changed After Cleanup: {mask_changed.sum():,}")

# ---- عينات قبل وبعد ----
print("\n=== أمثلة على التعديلات (قبل ⇨ بعد) ===")
samples = pd.DataFrame({
    "Before": _before[mask_changed].head(10),
    "After": df.loc[mask_changed, COL].head(10)
})
print(samples.to_string(index=False))



=== Column Cleanup: Normalized UNSPSC ===
🔹 Total Rows: 345,988
🔹 Number of Final Empty Values: 987
🔹 Number of Shortcodes Fixed (<8 digits): 18
🔹 Number of Values ​​Actually Changed After Cleanup: 18

=== أمثلة على التعديلات (قبل ⇨ بعد) ===
Before    After
401728 40172800
301817 30181700
401728 40172800
301817 30181700
301817 30181700
401733 40173300
401733 40173300
401733 40173300
301817 30181700
401729 40172900


In [139]:
print(df.shape)

(345988, 32)


In [140]:
import re
import pandas as pd

# keep-length per column
KEEP = {"Segment": 2, "Family": 4, "Class": 6}

def left_digits(val, n):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if not s:
        return None
    digits = re.sub(r"\D", "", s)  # keep digits only
    if not digits:
        return None
    # take the significant left part; if shorter than n, keep as-is
    return digits[:n] if len(digits) >= n else digits

# apply in-place
_before = df[list(KEEP.keys())].copy()

for col, n in KEEP.items():
    df[col] = df[col].apply(lambda v: left_digits(v, n)).astype("string")

# quick check
print("\n=== Trimmed UNSPSC levels (in place) ===")
for col, n in KEEP.items():
    lens = df[col].dropna().astype(str).str.len().value_counts().sort_index()
    changed = (( _before[col].astype(str) != df[col].astype(str) ) & df[col].notna()).sum()
    off_len = df[col].dropna().astype(str).str.len().ne(n).sum()
    print(f"{col:<7} keep={n} | changed rows={changed:,} | length≠{n}: {off_len:,}")
    print(lens.head(5))



=== Trimmed UNSPSC levels (in place) ===
Segment keep=2 | changed rows=342,723 | length≠2: 0
Segment
2    342723
Name: count, dtype: int64
Family  keep=4 | changed rows=342,723 | length≠4: 0
Family
4    342723
Name: count, dtype: int64
Class   keep=6 | changed rows=342,723 | length≠6: 0
Class
6    342723
Name: count, dtype: int64


In [141]:
import pandas as pd
import re


UNSPSC_COL = "Normalized UNSPSC"
SEG, FAM, CLS = "Segment", "Family", "Class"
SEG_T, FAM_T, CLS_T, COM_T = "Segment Title", "Family Title", "Class Title", "Commodity Title"


def safe_first_digits(val, n):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if not s:
        return None
    digits = re.sub(r"\D", "", s)
    if len(digits) < n:
        return None
    return digits[:n]


seg_derived = df[UNSPSC_COL].apply(lambda v: safe_first_digits(v, 2))
fam_derived = df[UNSPSC_COL].apply(lambda v: safe_first_digits(v, 4))
cls_derived = df[UNSPSC_COL].apply(lambda v: safe_first_digits(v, 6))

filled_seg_before = df[SEG].isna().sum()
filled_fam_before = df[FAM].isna().sum()
filled_cls_before = df[CLS].isna().sum()

df[SEG] = df[SEG].fillna(seg_derived).astype("string")
df[FAM] = df[FAM].fillna(fam_derived).astype("string")
df[CLS] = df[CLS].fillna(cls_derived).astype("string")

print("=== اشتقاق الأكواد من UNSPSC ===")
print(f"Segment filled: {filled_seg_before - df[SEG].isna().sum():,}")
print(f"Family  filled: {filled_fam_before - df[FAM].isna().sum():,}")
print(f"Class   filled: {filled_cls_before - df[CLS].isna().sum():,}")


def make_map(dataframe, code_col, title_col):

    return (
        dataframe[[code_col, title_col]]
        .dropna()
        .drop_duplicates()
        .set_index(code_col)[title_col]
        .to_dict()
    )

map_seg  = make_map(df, SEG,  SEG_T)
map_fam  = make_map(df, FAM,  FAM_T)
map_cls  = make_map(df, CLS,  CLS_T)
map_comm = make_map(df, UNSPSC_COL, COM_T)


def fill_titles_from_map(df, code_col, title_col, mapping):

    mapped = df[code_col].map(mapping)
    before_nulls = df[title_col].isna().sum()
    df[title_col] = df[title_col].where(df[title_col].notna(), mapped)
    filled = before_nulls - df[title_col].isna().sum()
    return filled

print("\n=== إسناد العناوين من نفس الجدول ===")
print(f"{SEG_T:<20}: +{fill_titles_from_map(df, SEG,  SEG_T,  map_seg):,}")
print(f"{FAM_T:<20}: +{fill_titles_from_map(df, FAM,  FAM_T,  map_fam):,}")
print(f"{CLS_T:<20}: +{fill_titles_from_map(df, CLS,  CLS_T,  map_cls):,}")
print(f"{COM_T:<20}: +{fill_titles_from_map(df, UNSPSC_COL, COM_T, map_comm):,}")


print("\n=== تحقق سريع بعد الإسناد ===")
for code_col, title_col in [(SEG, SEG_T), (FAM, FAM_T), (CLS, CLS_T), (UNSPSC_COL, COM_T)]:
    null_titles = df[title_col].isna().sum()
    print(f"{title_col:<20} | remaining nulls: {null_titles:,}")


for code_col, title_col in [(SEG, SEG_T), (FAM, FAM_T), (CLS, CLS_T), (UNSPSC_COL, COM_T)]:
    mask_filled = df[title_col].notna() & df[code_col].notna()
    if mask_filled.any():
        print(f"\n— after {code_col} → {title_col} —")
        print(
            df.loc[mask_filled, [code_col, title_col]]
              .drop_duplicates()
              .head(10)
              .to_string(index=False)
        )


=== اشتقاق الأكواد من UNSPSC ===
Segment filled: 2,278
Family  filled: 2,278
Class   filled: 2,278

=== إسناد العناوين من نفس الجدول ===
Segment Title       : +2,278
Family Title        : +2,058
Class Title         : +1,733
Commodity Title     : +0

=== تحقق سريع بعد الإسناد ===
Segment Title        | remaining nulls: 987
Family Title         | remaining nulls: 1,207
Class Title          | remaining nulls: 1,532
Commodity Title      | remaining nulls: 3,265

— أمثلة بعد الإسناد: Segment → Segment Title —
Segment                                                      Segment Title
     76                                       Industrial Cleaning Services
     44                      Office Equipment and Accessories and Supplies
     85                                                Healthcare Services
     40 Distribution and Conditioning Systems and Equipment and Components
     50                                 Food Beverage and Tobacco Products
     55                                 

In [142]:
import pandas as pd


UNSPSC = "Normalized UNSPSC"
SEG, FAM, CLS = "Segment", "Family", "Class"
SEG_T, FAM_T, CLS_T, COM_T = "Segment Title", "Family Title", "Class Title", "Commodity Title"

pairs = [
    (SEG, SEG_T),
    (FAM, FAM_T),
    (CLS, CLS_T),
    (UNSPSC, COM_T),
]

print("\n=== 🔍 فحص الطول والمطابقة مع UNSPSC ===")
for col, n in [(SEG,2),(FAM,4),(CLS,6)]:
    lens = df[col].dropna().astype(str).str.len()
    bad_len = (lens != n).sum()
    print(f"{col:<7} | القيم ذات طول ≠ {n}: {bad_len:,}")

mask_unspsc = df[UNSPSC].notna()
print("\n— مطابقة التجزئة مع UNSPSC —")
print(f"Segment == UNSPSC[:2]:", (df.loc[mask_unspsc, SEG].astype(str) == df.loc[mask_unspsc, UNSPSC].astype(str).str[:2]).all())
print(f"Family  == UNSPSC[:4]:", (df.loc[mask_unspsc, FAM].astype(str) == df.loc[mask_unspsc, UNSPSC].astype(str).str[:4]).all())
print(f"Class   == UNSPSC[:6]:", (df.loc[mask_unspsc, CLS].astype(str) == df.loc[mask_unspsc, UNSPSC].astype(str).str[:6]).all())


def make_map(df, code_col, title_col):
    """إنشاء خريطة كود → عنوان فريدة من الجدول."""
    return (
        df[[code_col, title_col]]
        .dropna()
        .drop_duplicates()
        .set_index(code_col)[title_col]
        .to_dict()
    )

print("\n=== 🔍 تحقق من اتساق الأكواد والعناوين (Code ↔ Title) ===")
for code_col, title_col in pairs:
    cmap = make_map(df, code_col, title_col)
    sub = df[[code_col, title_col]].dropna()
    mism = sub[title_col] != sub[code_col].map(cmap)
    n_bad = int(mism.sum())
    print(f"{code_col:<18} ↔ {title_col:<20} | تناقضات: {n_bad:,}")
    if n_bad:
        print(sub.loc[mism].head(5))


print("\n=== 📊 المتبقي من الفراغات في العناوين ===")
for code_col, title_col in pairs:
    nulls = int(df[title_col].isna().sum())
    filled = df[title_col].notna().sum()
    uniq_codes = df[code_col].dropna().nunique()
    print(f"{title_col:<20} | nulls={nulls:,} | filled={filled:,} | unique codes={uniq_codes:,}")

# أمثلة لأكواد بلا عنوان
for code_col, title_col in pairs:
    need = df[code_col].notna() & df[title_col].isna()
    if need.any():
        print(f"\n— 🔸 أكواد {code_col} بلا عنوان ({title_col}) —")
        print(df.loc[need, code_col].drop_duplicates().head(10).to_string(index=False))

print("\n=== 🧩Code ↔ Title ===")
for code_col, title_col in pairs:
    sub = df[[code_col, title_col]].dropna()
    n_per_code = sub.groupby(code_col)[title_col].nunique()
    n_conflict = int((n_per_code > 1).sum())
    print(f"{code_col:<18} ↔ {title_col:<20} | أكواد متعددة العناوين: {n_conflict:,}")
    if n_conflict:
        bad = n_per_code[n_per_code > 1].index[:10]
        for c in bad:
            titles = sub.loc[sub[code_col]==c, title_col].dropna().unique().tolist()
            print(f"  {c} → {titles}")

print("\n✅ done")



=== 🔍 فحص الطول والمطابقة مع UNSPSC ===
Segment | القيم ذات طول ≠ 2: 0
Family  | القيم ذات طول ≠ 4: 0
Class   | القيم ذات طول ≠ 6: 0

— مطابقة التجزئة مع UNSPSC —
Segment == UNSPSC[:2]: True
Family  == UNSPSC[:4]: True
Class   == UNSPSC[:6]: True

=== 🔍 تحقق من اتساق الأكواد والعناوين (Code ↔ Title) ===
Segment            ↔ Segment Title        | تناقضات: 0
Family             ↔ Family Title         | تناقضات: 931
     Family                                       Family Title
60     3118                                  Gaskets and seals
810    7215  lized trade construction and maintenance services
892    3118                                  Gaskets and seals
1132   3010             Structural components and basic shapes
2246   3010             Structural components and basic shapes
Class              ↔ Class Title          | تناقضات: 0
Normalized UNSPSC  ↔ Commodity Title      | تناقضات: 0

=== 📊 المتبقي من الفراغات في العناوين ===
Segment Title        | nulls=987 | filled=345,001 |

In [143]:
df[["Normalized UNSPSC", "Commodity Title", "Class", "Class Title",
    "Family", "Family Title", "Segment", "Segment Title"]].isna().sum()


,0
Normalized UNSPSC,987
Commodity Title,3265
Class,987
Class Title,1532
Family,987
Family Title,1207
Segment,987
Segment Title,987


In [144]:
cols = [
    "Classification Codes",
    "Normalized UNSPSC",
    "Commodity Title",
    "Class",
    "Class Title",
    "Family",
    "Family Title",
    "Segment",
    "Segment Title"
]

null_like_values = {
    "", " ", "NONE", "None", "N/A", "NA", "NULL",
    "NOT APPLICABLE", "UNKNOWN"
}

for col in cols:
    if col not in df.columns:
        continue
    s = df[col].astype(str).str.strip().str.upper()
    null_like_count = s.isin(null_like_values).sum()
    real_null_count = df[col].isna().sum()
    print(f"=== {col} ===")
    print(f"🔹 NaN/None  {real_null_count:,}")
    print(f"🔹like null : {null_like_count:,}")
    if null_like_count:
        print("most repeted")
        print(s[s.isin(null_like_values)].value_counts())
    print()


=== Classification Codes ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 987
أكثر القيم الشبيهة بالفراغ تكرارًا:
Classification Codes
NONE    987
Name: count, dtype: int64

=== Normalized UNSPSC ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 987
أكثر القيم الشبيهة بالفراغ تكرارًا:
Normalized UNSPSC
NONE    987
Name: count, dtype: int64

=== Commodity Title ===
🔹 NaN/None الحقيقية: 3,265
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Class ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Class Title ===
🔹 NaN/None الحقيقية: 1,532
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Family ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Family Title ===
🔹 NaN/None الحقيقية: 1,207
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Segment ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 0

=== Segment Title ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 0



In [145]:
import pandas as pd

cols_to_fix = ["Classification Codes", "Normalized UNSPSC"]


NULL_LIKE = {"", " ", "NONE", "None", "NULL", "N/A", "NA"}

for col in cols_to_fix:
    if col not in df.columns:
        continue

    def to_none(v):

        if isinstance(v, (list, dict)):
            return v

        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None

        s = str(v).strip()
        return None if s.upper() in NULL_LIKE else v

    df[col] = df[col].apply(to_none)


In [146]:
null_like_values = {"", " ", "NONE", "None", "N/A", "NA", "NULL", "NOT APPLICABLE", "UNKNOWN"}

for col in cols_to_fix:
    s_upper = df[col].astype(str).str.strip().str.upper()
    real_null = df[col].isna().sum()
    null_like = s_upper.isin(null_like_values).sum()
    print(f"=== {col} ===")
    print(f"🔹 NaN/None  {real_null:,}")
    print(f"🔹 like null {null_like:,}")
    if null_like:
        print(s_upper[s_upper.isin(null_like_values)].value_counts().head(10))
    print()


=== Classification Codes ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 987
Classification Codes
NONE    987
Name: count, dtype: int64

=== Normalized UNSPSC ===
🔹 NaN/None الحقيقية: 987
🔹 القيم النصية الشبيهة بالفراغ: 987
Normalized UNSPSC
NONE    987
Name: count, dtype: int64



In [147]:
import re
import pandas as pd
import numpy as np

col = "Location"


FLOAT_RE = r"-?\d{1,3}(?:\.\d+)?"
ZIP_RE   = r"\b\d{5}(?:-\d{4})?\b"

def has_coords(text):

    if not isinstance(text, str):
        return False
    m = re.search(r"\(" + FLOAT_RE + r"[, ]+" + FLOAT_RE + r"\)", text)
    return bool(m)

def is_zip_only(text):
    """يتحقق إن كان النص رقم ZIP فقط"""
    if not isinstance(text, str):
        return False
    has_zip = re.fullmatch(ZIP_RE, text.strip()) is not None
    has_paren = "(" in text or ")" in text
    return has_zip and not has_paren

# أقنعة
s = df[col].astype("string")
mask_null  = s.isna()
mask_coord = s.apply(has_coords)
mask_zip   = s.apply(is_zip_only)
mask_bad   = (~mask_null) & (~mask_coord) & (~mask_zip) & (s.str.strip() != "")



print("\n=== Initial check for Location column ===")
print(f"Total rows: {len(s):,}")
print(f"🔹 Actually empty (NaN/None): {mask_null.sum():,}")
print(f"🔹 Contains coordinates: {mask_coord.sum():,}")
print(f"🔹 Contains only ZIP: {mask_zip.sum():,}")
print(f"🔹 Useless strings: {mask_bad.sum():,}")

# عرض أمثلة
print("\n— sample —")
print(s[mask_bad].head(10))



=== فحص أولي لعمود Location ===
إجمالي الصفوف: 345,988
🔹 فارغ فعليًا (NaN/None): 70,107
🔹 يحتوي على إحداثيات: 272,679
🔹 يحتوي على ZIP فقط: 884
🔹 نصوص غير مفيدة: 2,318

— أمثلة من النصوص غير المفيدة —
96      7647

626     6416

855     3054

866     6484

868     8875

876     7647

942     7647

971     1821

993     3031

1033    7645

Name: Location, dtype: string


In [148]:
def extract_coords(text):
    """يستخرج الإحداثيات (lat, lon) إن وُجدت"""
    if not isinstance(text, str):
        return (np.nan, np.nan)
    m = re.search(r"\(\s*(" + FLOAT_RE + r")[,\s]+(" + FLOAT_RE + r")\s*\)", text)
    if m:
        lat, lon = float(m.group(1)), float(m.group(2))
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            return lat, lon
    return (np.nan, np.nan)


df["Latitude"], df["Longitude"] = zip(*df[col].map(extract_coords))


def clean_location(text):
    if pd.isna(text) or not str(text).strip():
        return None
    if has_coords(text):

        return re.sub(ZIP_RE, "", text).strip()
    if is_zip_only(text):
        return None
    return None

df[col] = df[col].map(clean_location).astype("string")

print("\n=== after===")
print(df[[col, "Latitude", "Longitude"]].head(10))



=== بعد التنظيف ===
                   Location   Latitude   Longitude
0                      <NA>        NaN         NaN
1                      <NA>        NaN         NaN
2  (38.662263, -121.346136)  38.662263 -121.346136
3  (38.580427, -121.494396)  38.580427 -121.494396
4  (38.580427, -121.494396)  38.580427 -121.494396
5  (36.193481, -119.358379)  36.193481 -119.358379
6  (38.580427, -121.494396)  38.580427 -121.494396
7  (34.379263, -118.547301)  34.379263 -118.547301
8  (34.073577, -118.145947)  34.073577 -118.145947
9   (42.506886, -83.407804)  42.506886  -83.407804


In [149]:

df.drop(columns=["Location"], inplace=True, errors="ignore")


print("Location موجود؟", "Location" in df.columns)


Location موجود؟ False


In [150]:
df["Latitude"] = (
    df["Latitude"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [151]:
df["Longitude"] = (
    df["Longitude"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
)


In [152]:
columns_to_drop = ["po_normalized", "req_normalized"]
df = df.drop(columns=columns_to_drop, errors='ignore')

In [153]:

df.to_csv("purchases_clean.csv", index=False, encoding="utf-8")
print("✅ saved: purchases_clean.csv")


✅ saved: purchases_clean.csv


In [154]:
print(df.head())

  Creation Date Purchase Date Fiscal Year   LPA Number Purchase Order Number  \
0    2013-08-27           NaT   2013-2014   7-12-70-26            req0011118   
1    2014-01-29           NaT   2013-2014         None            req0011932   
2    2013-11-01           NaT   2013-2014         None            req0011476   
3    2014-03-12    2014-03-12   2013-2014  1-10-75-60A            4500221028   
4    2014-10-10           NaT   2014-2015  1-14-75-60A            req0013911   

  Requisition Number Acquisition Type Sub-Acquisition Type  \
0         req0011118         IT Goods                 None   
1         req0011932     NON-IT Goods                 None   
2         req0011476      IT Services                 None   
3                        NON-IT Goods                 None   
4         req0013911     NON-IT Goods                 None   

     Acquisition Method Sub-Acquisition Method  ... Commodity Title   Class  \
0             WSCA/Coop                   None  ...             NaN

In [155]:
!pip install pymongo


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.0 MB/s eta 0:00:00


In [156]:
import pandas as pd
import datetime
from bson import json_util
import json


date_cols = ["Creation Date", "Purchase Date"]


list_columns = ["Classification Codes", "Supplier Qualifications"]

def safe_null_cleaner(row):
    for col in row.index:
        if col in list_columns:
            continue
        try:
            if pd.isna(row[col]):
                row[col] = None
        except Exception:
            pass
    return row


df = df.apply(safe_null_cleaner, axis=1)

def fix_only_valid_datetimes(record):
    for col in date_cols:
        val = record.get(col)
        try:

            if isinstance(val, (pd.Timestamp, datetime.datetime)):

                if pd.isna(val):
                    record[col] = None
                else:

                    record[col] = val.tz_convert("UTC") if val.tzinfo else val.tz_localize("UTC")
            else:
                record[col] = None
        except Exception:
            record[col] = None
    return record



records = df.to_dict(orient="records")

with open("mongo_readyt2.json", "w", encoding="utf-8") as f:
    for i, rec in enumerate(records):
        try:
            clean = fix_only_valid_datetimes(rec)
            json_line = json_util.dumps(clean)
            f.write(json_line + "\n")
        except Exception as e:
            print(f"\n🚨 خطأ في السطر {i}: {e}")
            print("📄 السجل:")
            for k, v in rec.items():
                print(f"  - {k} ({type(v)}): {repr(v)}")
            break



In [ ]:
df.to_json("cleaned_data.ndjson", orient="records", lines=True, force_ascii=False)


In [157]:
print(df.shape)

(345988, 33)


In [158]:
print(df.head())

  Creation Date Purchase Date Fiscal Year   LPA Number Purchase Order Number  \
0    2013-08-27           NaT   2013-2014   7-12-70-26            req0011118   
1    2014-01-29           NaT   2013-2014         None            req0011932   
2    2013-11-01           NaT   2013-2014         None            req0011476   
3    2014-03-12    2014-03-12   2013-2014  1-10-75-60A            4500221028   
4    2014-10-10           NaT   2014-2015  1-14-75-60A            req0013911   

  Requisition Number Acquisition Type Sub-Acquisition Type  \
0         req0011118         IT Goods                 None   
1         req0011932     NON-IT Goods                 None   
2         req0011476      IT Services                 None   
3                        NON-IT Goods                 None   
4         req0013911     NON-IT Goods                 None   

     Acquisition Method Sub-Acquisition Method  ... Commodity Title   Class  \
0             WSCA/Coop                   None  ...            None

In [159]:
list(df.columns)

['Creation Date',
 'Purchase Date',
 'Fiscal Year',
 'LPA Number',
 'Purchase Order Number',
 'Requisition Number',
 'Acquisition Type',
 'Sub-Acquisition Type',
 'Acquisition Method',
 'Sub-Acquisition Method',
 'Department Name',
 'Supplier Code',
 'Supplier Name',
 'Supplier Qualifications',
 'Supplier Zip Code',
 'CalCard',
 'Item Name',
 'Item Description',
 'Quantity',
 'Unit Price',
 'Total Price',
 'Classification Codes',
 'Normalized UNSPSC',
 'Commodity Title',
 'Class',
 'Class Title',
 'Family',
 'Family Title',
 'Segment',
 'Segment Title',
 'LineType',
 'Latitude',
 'Longitude']